# SFT Training
**Base model** - Qwen2.5-1.5B
**Data** - Ultrafeedback Binarized ~ 5k samples
**Technique** - QLoRA 4-bit + SFTTrainer

## 1-Install Libraries

In [ ]:
!pip install -q --force-reinstall --no-deps bitsandbytes==0.43.1

In [ ]:
!pip install -q --force-reinstall --no-deps transformers==4.44.2

In [ ]:
!pip install -q --force-reinstall --no-deps peft==0.12.0

In [ ]:
!pip install -q --force-reinstall --no-deps trl==0.11.4

In [ ]:
!pip install -q --force-reinstall --no-deps accelerate==0.34.2

In [ ]:
!pip uninstall -y bitsandbytes
!pip install -U bitsandbytes

## Restart and Clear outputs and then run the next cell

In [1]:
import transformers, trl, peft, bitsandbytes, accelerate
print(f"transformers  : {transformers.__version__}")
print(f"trl           : {trl.__version__}")
print(f"peft          : {peft.__version__}")
print(f"bitsandbytes  : {bitsandbytes.__version__}")
print(f"accelerate    : {accelerate.__version__}")

transformers  : 4.44.2
trl           : 0.11.4
peft          : 0.12.0
bitsandbytes  : 0.49.2
accelerate    : 0.34.2


## 2-Imports

In [2]:
import torch
import json
from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainerCallback
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)} — {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")

2026-06-10 07:05:35.011886: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781075135.224617     358 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781075135.286033     358 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781075135.813007     358 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781075135.813051     358 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781075135.813054     358 computation_placer.cc:177] computation placer alr

Torch: 2.10.0+cu128
CUDA available: True
  GPU 0: Tesla T4 — 15.6 GB
  GPU 1: Tesla T4 — 15.6 GB


## 3- Setting Configs

In [9]:
# ── Model ────────────────────────────────────────────────────────────────────
MODEL_NAME  = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR  = "/kaggle/working/sft_model"
DATA_PATH   = "/kaggle/input/datasets/aditig99/sft-dataset/sft_train.jsonl"
MAX_LENGTH  = 512

# ── LoRA — key change from before: higher rank for better expressiveness ─────
LORA_R          = 16      
LORA_ALPHA      = 32       
LORA_DROPOUT    = 0.05     
LORA_TARGET     = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# ── Training ─────────────────────────────────────────────────────────────────
NUM_EPOCHS      = 3       
BATCH_SIZE      = 2        
GRAD_ACCUM      = 8       
LR              = 2e-4
LR_SCHEDULER    = "cosine" 
WARMUP_RATIO    = 0.05     
WEIGHT_DECAY    = 0.01
MAX_GRAD_NORM   = 0.3      

# ── Eval ─────────────────────────────────────────────────────────────────────
EVAL_SPLIT      = 0.05     
EVAL_STEPS      = 50
SAVE_STEPS      = 100
LOGGING_STEPS   = 10

print("Config set")
print(f"  LoRA rank: {LORA_R} (was 8 before)")
print(f"  Epochs: {NUM_EPOCHS} (was 1 before)")
print(f"  Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")
print(f"  Target modules: {LORA_TARGET}")

Config set
  LoRA rank: 16 (was 8 before)
  Epochs: 3 (was 1 before)
  Effective batch size: 16
  Target modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']


## Data Overview

In [10]:
import json
import pandas as pd

def load_jsonl(file):
    with open(file) as f:
        return [json.loads(line) for line in f]

sft    = pd.DataFrame(load_jsonl(DATA_PATH))
print(f"SFT:    {sft.shape}")

print("\nSFT sample:"); display(sft.head())

SFT:    (4242, 3)

SFT sample:


,text,prompt,response
0,"<|im_start|>system\nYou are Qwen, created by A...","Make a meme title with the name ""Greta Thunber...","""Greta Thunberg Discovers the Most Climate-Des..."
1,"<|im_start|>system\nYou are Qwen, created by A...",Determine the type of quadrilateral formed by ...,Hello! I'd be happy to help you determine the ...
2,"<|im_start|>system\nYou are Qwen, created by A...",You are given a sentence in Spanish. Your job ...,بچه‌ها، بر اساس قوانین، شما معلمان بزرگ افسانه...
3,"<|im_start|>system\nYou are Qwen, created by A...",what are the minumum requirements for a ubuntu...,The minimum requirements for an Ubuntu ELK sta...
4,"<|im_start|>system\nYou are Qwen, created by A...","Given a sentence in the Japanese, provide an e...",ความเจ็บป่วยอาจจะส่งผลต่อโอกาสของเธอ แต่ในสัมภ...


## 4- Load & Prepare Dataset

In [11]:
# Load from jsonl
raw = []
with open(DATA_PATH) as f:
    for line in f:
        raw.append(json.loads(line))

print(f"Loaded {len(raw)} samples")
print(f"Columns: {list(raw[0].keys())}")

# Convert to HF Dataset
dataset = Dataset.from_list(raw)

# Train/eval split
split    = dataset.train_test_split(test_size=EVAL_SPLIT, seed=42)
train_ds = split["train"]
eval_ds  = split["test"]

print(f"Train: {len(train_ds)} | Eval: {len(eval_ds)}")
print(f"\nSample text preview:")
print(train_ds[0]["text"][:300])

Loaded 4242 samples
Columns: ['text', 'prompt', 'response']
Train: 4029 | Eval: 213

Sample text preview:
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Catholic history that I can share on facebook for my followers<|im_end|>
<|im_start|>assistant
Title: "Did you know? The Great Schism of 1054 and the birth of Eastern and Western Church


## 5- Load Tokenizer

In [12]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token     = tokenizer.eos_token
tokenizer.padding_side  = "right"  

print(f"Vocab size:    {tokenizer.vocab_size}")
print(f"Pad token:     {tokenizer.pad_token}")
print(f"Padding side:  {tokenizer.padding_side}")

# quick token length check on the dataset
lengths = [len(tokenizer(s["text"], truncation=False)["input_ids"]) for s in train_ds.select(range(200))]
print(f"\nToken length stats (first 200 samples):")
print(f"  min: {min(lengths)}")
print(f"  max: {max(lengths)}")
print(f"  avg: {sum(lengths)//len(lengths)}")
print(f"  over MAX_LENGTH ({MAX_LENGTH}): {sum(1 for l in lengths if l > MAX_LENGTH)}")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vocab size:    151643
Pad token:     <|im_end|>
Padding side:  right

Token length stats (first 200 samples):
  min: 54
  max: 504
  avg: 287
  over MAX_LENGTH (512): 0


## 6- Load Model with QLoRA

In [13]:
# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit               = True,
    bnb_4bit_quant_type        = "nf4",
    bnb_4bit_compute_dtype     = torch.bfloat16,   
    bnb_4bit_use_double_quant  = True,             
)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config = bnb_config,
    device_map          = "auto",
    trust_remote_code   = True,
    torch_dtype         = torch.bfloat16,
)
model.config.use_cache = False                    
model.config.pretraining_tp = 1

# count total vs trainable params
total  = sum(p.numel() for p in model.parameters())
print(f"Model loaded")
print(f"Total params: {total/1e6:.1f}M")

Loading model...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded
Total params: 888.6M


In [14]:
from peft import prepare_model_for_kbit_training
model = prepare_model_for_kbit_training(model)
print("Model prepared for kbit training")

Model prepared for kbit training


## 7-Apply LoRA

In [15]:
lora_config = LoraConfig(
    r                   = LORA_R,
    lora_alpha          = LORA_ALPHA,
    lora_dropout        = LORA_DROPOUT,
    target_modules      = LORA_TARGET,
    bias                = "none",
    task_type           = TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable/1e6:.2f}M / {total/1e6:.1f}M ({100*trainable/total:.2f}%)")
model.print_trainable_parameters()

Trainable params: 18.46M / 907.1M (2.04%)
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


## 8 - Training Configs

In [16]:
sft_config = SFTConfig(
    # output
    output_dir                  = OUTPUT_DIR,

    # data
    max_seq_length              = MAX_LENGTH,
    dataset_text_field          = "text",         
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    gradient_checkpointing      = True,            

    # optimizer
    learning_rate               = LR,
    lr_scheduler_type           = LR_SCHEDULER,
    warmup_ratio                = WARMUP_RATIO,
    weight_decay                = WEIGHT_DECAY,
    max_grad_norm               = MAX_GRAD_NORM,
    optim                       = "paged_adamw_8bit",  
    # precision
    bf16                        = True,
    fp16                        = False,

    # eval & saving
    eval_strategy               = "steps",
    eval_steps                  = EVAL_STEPS,
    save_strategy               = "steps",
    save_steps                  = SAVE_STEPS,
    save_total_limit            = 2,              
    load_best_model_at_end      = True,           
    metric_for_best_model       = "eval_loss",

    # logging
    logging_steps               = LOGGING_STEPS,
    report_to                   = "none",

    # packing — off
    packing                     = False,
)

print("SFTConfig set")

SFTConfig set


## 9-Loss Logging CallBack

In [17]:
# simple callback to print train + eval loss at each eval step
# so you can see if model is converging without wandb
class LossLogger(TrainerCallback):
    def __init__(self):
        self.train_losses = []
        self.eval_losses  = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return
        step = state.global_step
        if "loss" in logs:
            self.train_losses.append((step, logs["loss"]))
            print(f"  Step {step:>4} | train_loss: {logs['loss']:.4f}", end="")
        if "eval_loss" in logs:
            self.eval_losses.append((step, logs["eval_loss"]))
            print(f"  |  eval_loss: {logs['eval_loss']:.4f}", end="")
        print()

loss_logger = LossLogger()
print("Callback ready")

Callback ready


## 10 - Train

In [18]:
trainer = SFTTrainer(
    model           = model,
    args            = sft_config,
    train_dataset   = train_ds,
    eval_dataset    = eval_ds,
    tokenizer       = tokenizer,
    callbacks       = [loss_logger],
)

print("Starting SFT training...")
print(f"  Total steps: {trainer.args.max_steps if trainer.args.max_steps > 0 else 'auto'}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Effective batch size: {BATCH_SIZE * GRAD_ACCUM}\n")

trainer.train()

print("\nTraining complete")

Map:   0%|          | 0/4029 [00:00<?, ? examples/s]

Map:   0%|          | 0/213 [00:00<?, ? examples/s]

Starting SFT training...
  Total steps: auto
  Epochs: 3
  Effective batch size: 16



/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept alive. This can happen if the autograd graph is still being kept alive by tensors such a

Step,Training Loss,Validation Loss
50,1.342500,1.278455
100,1.231400,1.250671
150,1.283800,1.235069
200,1.221700,1.225723
250,1.310000,1.215748
300,1.183700,1.217093
350,1.156500,1.215482
400,1.162100,1.213251
450,1.085100,1.211377
500,1.161800,1.209590


...

  Step  510 | train_loss: 1.0695
  Step  520 | train_loss: 1.0584
  Step  530 | train_loss: 1.0273
  Step  540 | train_loss: 1.0350
  Step  550 | train_loss: 1.0267
  |  eval_loss: 1.2270
  Step  560 | train_loss: 1.0598
  Step  570 | train_loss: 1.0500
  Step  580 | train_loss: 1.0315
  Step  590 | train_loss: 1.0270
  Step  600 | train_loss: 1.0196
  |  eval_loss: 1.2269


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


  Step  610 | train_loss: 1.0934
  Step  620 | train_loss: 0.9704
  Step  630 | train_loss: 1.0520
  Step  640 | train_loss: 1.1050
  Step  650 | train_loss: 1.0363
  |  eval_loss: 1.2267
  Step  660 | train_loss: 1.0002
  Step  670 | train_loss: 0.9566
  Step  680 | train_loss: 1.1035
  Step  690 | train_loss: 0.9704
  Step  700 | train_loss: 0.9513
  |  eval_loss: 1.2263


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


  Step  710 | train_loss: 1.0635
  Step  720 | train_loss: 1.0408
  Step  730 | train_loss: 0.9908
  Step  740 | train_loss: 1.0418
  Step  750 | train_loss: 1.0202
  |  eval_loss: 1.2266


Training complete


## 11- Save

In [20]:
# saves only the LoRA adapter — ~80-100MB, not the full model
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

import os
size_mb = sum(os.path.getsize(os.path.join(OUTPUT_DIR, f))
              for f in os.listdir(OUTPUT_DIR)) / 1e6

print(f"Adapter saved to: {OUTPUT_DIR}")
print(f"Total size: {size_mb:.1f} MB")
print(f"Files: {os.listdir(OUTPUT_DIR)}")

Adapter saved to: /kaggle/working/sft_model
Total size: 85.4 MB
Files: ['checkpoint-500', 'tokenizer.json', 'special_tokens_map.json', 'README.md', 'adapter_model.safetensors', 'tokenizer_config.json', 'vocab.json', 'adapter_config.json', 'checkpoint-753', 'added_tokens.json', 'merges.txt', 'training_args.bin']


In [21]:
import shutil

shutil.make_archive(
    "/kaggle/working/sft_model_zip",
    "zip",
    "/kaggle/working/sft_model"  # your model folder
)

'/kaggle/working/sft_model_zip.zip'

## 12 - Check Loss Curve

In [22]:
print("── Loss Summary ──")
if loss_logger.train_losses:
    first_loss = loss_logger.train_losses[0][1]
    last_loss  = loss_logger.train_losses[-1][1]
    print(f"Train loss:  {first_loss:.4f} → {last_loss:.4f}  (drop: {first_loss - last_loss:.4f})")

if loss_logger.eval_losses:
    best_eval = min(loss_logger.eval_losses, key=lambda x: x[1])
    last_eval = loss_logger.eval_losses[-1]
    print(f"Eval loss:   best={best_eval[1]:.4f} at step {best_eval[0]} | final={last_eval[1]:.4f}")

# overfitting check
if loss_logger.eval_losses:
    best_step = min(loss_logger.eval_losses, key=lambda x: x[1])[0]
    total_steps = loss_logger.eval_losses[-1][0]
    if best_step < total_steps * 0.6:
        print("\nBest eval loss was early in training — possible overfitting. Check last vs best eval loss.")
    else:
        print("\nLoss curve looks healthy.")

── Loss Summary ──
Train loss:  1.8001 → 1.0202  (drop: 0.7799)
Eval loss:   best=1.2096 at step 500 | final=1.2266

Loss curve looks healthy.


## 13 - Quick Inference Test

In [23]:
# test the trained model on a few prompts
# compare mentally with what the base model would say

model.eval()

test_prompts = [
    "Explain the difference between supervised and unsupervised learning in simple terms.",
    "What are three good habits for improving focus while studying?",
    "Write a short Python function to reverse a string."
]

for prompt in test_prompts:
    messages = [{"role": "user", "content": prompt}]
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens    = 200,
            temperature       = 0.7,
            top_p             = 0.9,
            do_sample         = True,
            pad_token_id      = tokenizer.eos_token_id,
            repetition_penalty= 1.1,   # reduces repetition
        )

    # decode only the new tokens, not the prompt
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response   = tokenizer.decode(new_tokens, skip_special_tokens=True)

    print(f"Prompt:   {prompt}")
    print(f"Response: {response}")
    print("-" * 60)

Prompt:   Explain the difference between supervised and unsupervised learning in simple terms.
Response: Sure! I'd be happy to help you understand the difference between supervised and unsupervised learning.

Supervised learning is a type of machine learning where the model learns from labeled data, which means that each example has an associated label or target variable. The goal of the model is to learn the mapping between input features (X) and output labels (y). For instance, if we have a dataset with images of cats and dogs, the model will learn to classify these images into two categories: "cat" or "dog". This type of learning requires human-supplied labels for the training examples.

On the other hand, unsupervised learning is a type of machine learning where the model learns from unlabeled data, without any prior knowledge of the desired outcome or class labels. The objective of unsupervised learning algorithms is to discover patterns, structure, or relationships within the dat

## 14 - Win Rate Evalaution vs Base Model

In [25]:
!pip install -q rouge-score

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [30]:
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer, util
import numpy as np
import torch

embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

def generate_response(model, tokenizer, prompt):
    messages = [{"role": "user", "content": prompt}]

    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [32]:
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

base_model.eval()

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2SdpaAttention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
          (rotary_emb): Qwen2RotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qw

In [33]:
sft_model = model
sft_tokenizer = tokenizer

In [35]:
print(type(test_samples[0]["chosen"]))
print(test_samples[0]["chosen"])

<class 'str'>
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
You will be given a definition of a task first, then some input of the task.
In this task, you're given a statement and three sentences as choices. Your job is to determine which sentence can be inferred from the statement. Incorrect choices change the meaning in important ways or have details that are not mentioned in the statement. Indicate your answer as 1,2, or 3 corresponding to the choice number of the selected sentence.

Statement: King said she is accused of having too much clutter. Choices:  1. Clutter is a problem for King. 2. King is a neat freak. 3. King has not thrown anything out for over five years.
Output:<|im_end|>
<|im_start|>assistant
The correct answer is 1. Clutter is a problem for King.

Explanation: The statement "King said she is accused of having too much clutter" implies that King has a lot of clutter and that it is a problem for her

In [37]:
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer, util
import numpy as np
import torch

# -----------------------------
# Load embedding model
# -----------------------------
embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

# -----------------------------
# Extract assistant response
# -----------------------------
def extract_assistant(text):
    marker = "<|im_start|>assistant"

    if marker in text:
        text = text.split(marker, 1)[1]

    text = text.replace("<|im_end|>", "")

    return text.strip()

# -----------------------------
# Generation function
# -----------------------------
def generate_response(model, tokenizer, prompt):

    messages = [
        {"role": "user", "content": prompt}
    ]

    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1
        )

    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

# -----------------------------
# Evaluation
# -----------------------------
base_scores = []
sft_scores = []

wins = 0
losses = 0
ties = 0

from tqdm.auto import tqdm

for idx, sample in enumerate(tqdm(test_samples)):

    prompt = sample["prompt"]

    reference = extract_assistant(
        sample["chosen"]
    )

    # Generate
    base_resp = generate_response(
        base_model,
        base_tokenizer,
        prompt
    )

    sft_resp = generate_response(
        sft_model,
        sft_tokenizer,
        prompt
    )

    # Embeddings
    ref_emb = embedder.encode(
        reference,
        convert_to_tensor=True
    )

    base_emb = embedder.encode(
        base_resp,
        convert_to_tensor=True
    )

    sft_emb = embedder.encode(
        sft_resp,
        convert_to_tensor=True
    )

    # Similarities
    base_sim = util.cos_sim(
        ref_emb,
        base_emb
    ).item()

    sft_sim = util.cos_sim(
        ref_emb,
        sft_emb
    ).item()

    base_scores.append(base_sim)
    sft_scores.append(sft_sim)

    # Win count
    if sft_sim > base_sim:
        wins += 1
    elif base_sim > sft_sim:
        losses += 1
    else:
        ties += 1

    if (idx + 1) % 10 == 0:
        print(f"Processed {idx+1}/{len(test_samples)}")

# -----------------------------
# Results
# -----------------------------
avg_base = np.mean(base_scores)
avg_sft = np.mean(sft_scores)

improvement = (
    (avg_sft - avg_base)
    / avg_base
) * 100

win_rate = wins / (wins + losses)

print("\n" + "="*50)
print("FINAL RESULTS")
print("="*50)

print(f"Base Similarity : {avg_base:.4f}")
print(f"SFT Similarity  : {avg_sft:.4f}")
print(f"Improvement     : {improvement:.2f}%")

print()
print(f"SFT Wins : {wins}")
print(f"Base Wins: {losses}")
print(f"Ties     : {ties}")
print(f"SFT Win Rate: {win_rate:.2%}")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


  0%|          | 0/100 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


Processed 10/100
Processed 20/100
Processed 30/100
Processed 40/100
Processed 50/100
Processed 60/100
Processed 70/100
Processed 80/100
Processed 90/100
Processed 100/100

FINAL RESULTS
Base Similarity : 0.6212
SFT Similarity  : 0.6898
Improvement     : 11.03%

SFT Wins : 65
Base Wins: 35
Ties     : 0
SFT Win Rate: 65.00%
